# Snowflake Marketplace Listings

The **Snowflake Marketplace** is the distribution platform for data products — datasets, data services, and native apps.  
It supports two primary listing types:

| Type | Visibility | Approval |
|------|-----------|----------|
| **Standard (Public)** | All Snowflake customers | Requires Snowflake review |
| **Personalized (Private)** | Specified accounts only | Publish directly |

**Auto-fulfillment** handles cross-region delivery automatically — the provider pays for replication.

This notebook walks through provider-side setup: preparing data, creating shares, and monitoring listing telemetry.

## 1. Setup — Provider Data

In [ ]:
USE ROLE ACCOUNTADMIN;
CREATE OR REPLACE DATABASE marketplace_demo;
CREATE SCHEMA marketplace_demo.curated;
CREATE SCHEMA marketplace_demo.raw;

USE ROLE SECURITYADMIN;
CREATE ROLE IF NOT EXISTS share_admin;
CREATE ROLE IF NOT EXISTS share_monitor;
GRANT ROLE share_admin TO ROLE SYSADMIN;
GRANT ROLE share_monitor TO ROLE share_admin;
GRANT CREATE SHARE ON ACCOUNT TO ROLE share_admin;
GRANT MANAGE SHARE TARGET ON ACCOUNT TO ROLE share_admin;
GRANT USAGE ON DATABASE marketplace_demo TO ROLE share_admin;
GRANT USAGE ON ALL SCHEMAS IN DATABASE marketplace_demo TO ROLE share_admin;
GRANT SELECT ON ALL TABLES IN SCHEMA marketplace_demo.raw TO ROLE share_admin;
GRANT CREATE VIEW ON SCHEMA marketplace_demo.curated TO ROLE share_admin;
GRANT IMPORTED PRIVILEGES ON DATABASE snowflake TO ROLE share_monitor;

USE ROLE ACCOUNTADMIN;
CREATE TABLE marketplace_demo.raw.weather_metrics AS
SELECT n_nationkey AS station_id,
       n_name AS station_name,
       n_regionkey AS region_id,
       MOD(n_nationkey * 7, 40) - 10 AS temperature_celsius,
       MOD(n_nationkey * 13, 100) AS humidity_pct,
       CURRENT_DATE() - MOD(n_nationkey, 365) AS measurement_date
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.NATION;

GRANT SELECT ON ALL TABLES IN SCHEMA marketplace_demo.raw TO ROLE share_admin;

## 2. Create Secure Views for Listing

In [ ]:
USE ROLE share_admin;

CREATE OR REPLACE SECURE VIEW marketplace_demo.curated.v_weather_summary AS
SELECT station_name, region_id, temperature_celsius, humidity_pct, measurement_date
FROM marketplace_demo.raw.weather_metrics;

CREATE OR REPLACE SECURE VIEW marketplace_demo.curated.v_regional_averages AS
SELECT region_id, AVG(temperature_celsius) AS avg_temp, AVG(humidity_pct) AS avg_humidity
FROM marketplace_demo.raw.weather_metrics
GROUP BY region_id;

## 3. Create Share (Foundation for Listing)

Every marketplace listing is backed by a **share** (or application package).

In [ ]:
USE ROLE share_admin;

CREATE OR REPLACE SHARE marketplace_weather_share
    COMMENT = 'Weather metrics for marketplace listing';
GRANT USAGE ON DATABASE marketplace_demo TO SHARE marketplace_weather_share;
GRANT USAGE ON SCHEMA marketplace_demo.curated TO SHARE marketplace_weather_share;
GRANT SELECT ON ALL VIEWS IN SCHEMA marketplace_demo.curated TO SHARE marketplace_weather_share;

DESCRIBE SHARE marketplace_weather_share;

## 4. Creating a Listing (Snowsight UI)

Marketplace listings are created via the **Provider Studio** in Snowsight. Steps:

1. Navigate to **Data Products** → **Provider Studio**
2. Click **+ Listing**
3. Select listing type:
   - **Standard (Public):** Visible to all Snowflake customers
   - **Personalized (Private):** Visible only to specified accounts
4. Associate the share: Select `MARKETPLACE_WEATHER_SHARE`
5. Add metadata: title, description, sample queries, data dictionary
6. For cross-region delivery, enable **Auto-Fulfillment**
7. Submit for review (public listings) or publish directly (private)

> **Note:** Programmatic listing management is available via the Snowflake REST API.

## 5. Auto-Fulfillment (Cross-Region)

Auto-fulfillment replicates data to consumer regions automatically. Provider pays for replication.

In [ ]:
-- Trigger an on-demand refresh after ETL completes (if auto-fulfillment is enabled)
-- SELECT SYSTEM$TRIGGER_LISTING_REFRESH('<listing_global_name>');

-- Monitor cross-region data transfer
SELECT
    start_time, source_region, target_region,
    bytes_transferred, transfer_type
FROM SNOWFLAKE.ACCOUNT_USAGE.DATA_TRANSFER_HISTORY
WHERE transfer_type = 'REPLICATION'
  AND start_time >= DATEADD('day', -30, CURRENT_TIMESTAMP())
ORDER BY start_time DESC;

## 6. Monitor Listing Telemetry

In [ ]:
USE ROLE share_monitor;

-- Daily listing telemetry
SELECT
    event_date,
    event_type,
    listing_name,
    listing_display_name,
    consumer_accounts_daily
FROM SNOWFLAKE.DATA_SHARING_USAGE.LISTING_TELEMETRY_DAILY
WHERE event_date >= DATEADD('day', -30, CURRENT_DATE())
ORDER BY event_date DESC;

-- Listing access history (who installed)
SELECT
    listing_name,
    listing_display_name,
    consumer_account_locator,
    event_date,
    event_type
FROM SNOWFLAKE.DATA_SHARING_USAGE.LISTING_ACCESS_HISTORY
ORDER BY event_date DESC;

## 7. Teardown

In [ ]:
USE ROLE ACCOUNTADMIN;
DROP SHARE IF EXISTS marketplace_weather_share;
DROP DATABASE IF EXISTS marketplace_demo;
DROP ROLE IF EXISTS share_admin;
DROP ROLE IF EXISTS share_monitor;

## References

- [Snowflake Marketplace Overview](https://docs.snowflake.com/en/user-guide/data-marketplace)
- [About Listings](https://docs.snowflake.com/en/user-guide/data-marketplace-listings)
- [Provider Studio](https://docs.snowflake.com/en/user-guide/data-marketplace-provider)
- [Auto-Fulfillment](https://docs.snowflake.com/en/user-guide/data-marketplace-auto-fulfillment)
- [LISTING_TELEMETRY_DAILY View](https://docs.snowflake.com/en/sql-reference/data-sharing-usage/listing_telemetry_daily)
- [LISTING_ACCESS_HISTORY View](https://docs.snowflake.com/en/sql-reference/data-sharing-usage/listing_access_history)
- [Personalized Listings](https://docs.snowflake.com/en/user-guide/data-marketplace-personalized-listings)
- [CREATE SHARE](https://docs.snowflake.com/en/sql-reference/sql/create-share)